<a href="https://colab.research.google.com/github/joaomerjam/ECON3916-33674-Statistical-Machine-Learning/blob/main/Lab%2013/Lab_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# Step 1: Ingestion and Naive Model
url = 'https://raw.githubusercontent.com/joaomerjam/ECON3916-33674-Statistical-Machine-Learning/refs/heads/main/Lab%2013/Data/Zillow_California_2026_Hedonic.csv'
df = pd.read_csv(url)
df.head()

,Property_Age,Distance_to_Tech_Hub,Sale_Price
0,77.5,38.1,684100.56
1,11.0,95.1,413634.22
2,47.7,73.5,456709.35
3,61.9,60.3,624533.95
4,100.8,16.4,870137.54


In [18]:
naive_model = smf.ols('Sale_Price ~ Property_Age', data=df).fit()
print(naive_model.summary())
print("\nNaive Age Coefficient:", naive_model.params['Property_Age'])


                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                     3105.
Date:                Wed, 18 Mar 2026   Prob (F-statistic):          1.26e-308
Time:                        20:09:02   Log-Likelihood:                -12818.
No. Observations:                1000   AIC:                         2.564e+04
Df Residuals:                     998   BIC:                         2.565e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     3.013e+05   7218.570     41.742   

In [19]:
# Step 2: The Multivariate Model
naive_model = smf.ols('Sale_Price ~ Property_Age + Distance_to_Tech_Hub', data=df).fit()
print(naive_model.summary())
print("\nNaive Age Coefficient:", naive_model.params['Property_Age'])


                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.954
Model:                            OLS   Adj. R-squared:                  0.954
Method:                 Least Squares   F-statistic:                 1.040e+04
Date:                Wed, 18 Mar 2026   Prob (F-statistic):               0.00
Time:                        20:09:02   Log-Likelihood:                -11982.
No. Observations:                1000   AIC:                         2.397e+04
Df Residuals:                     997   BIC:                         2.399e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept             1.203e+06 

In [20]:
# Step 3: FWL Theorem Manual Proof
# 3a: Partial out distance from Price
res_y_model = smf.ols('Sale_Price ~ Distance_to_Tech_Hub', data=df).fit()
df['Price_Residuals'] = res_y_model.resid


In [21]:
# 3b: Partial out distance from Age
res_x_model = smf.ols('Property_Age ~ Distance_to_Tech_Hub', data=df).fit()
df['Age_Residuals'] = res_x_model.resid

In [22]:
# 3c: Regress Residuals on Residuals (-1 removes the intercept for exact mathematical matching)
fwl_model = smf.ols('Price_Residuals ~ Age_Residuals', data=df).fit()
print("\nFWL Isolated Age Coefficient:", fwl_model.params['Age_Residuals'])


FWL Isolated Age Coefficient: -2063.129216802138


In [23]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go

# ── 1. Simulate the dataset ──────────────────────────────────────────────────
np.random.seed(42)
n = 200

property_age       = np.random.uniform(0, 50, n)     # years old
distance_to_tech   = np.random.uniform(0.5, 30, n)   # km to hub

# True DGP: price rises with newer homes, falls with distance
sale_price = (
    450_000
    - 3_200 * property_age
    - 8_500 * distance_to_tech
    + np.random.normal(0, 25_000, n)
)

df = pd.DataFrame({
    "Sale_Price":         sale_price,
    "Property_Age":       property_age,
    "Distance_to_Tech_Hub": distance_to_tech,
})

# ── 2. OLS regression ────────────────────────────────────────────────────────
X = sm.add_constant(df[["Property_Age", "Distance_to_Tech_Hub"]])
model = sm.OLS(df["Sale_Price"], X).fit()
print(model.summary())

# ── 3. Extract coefficients ──────────────────────────────────────────────────
# statsmodels stores coefficients in model.params, indexed by variable name.
# model.params["const"]                  → intercept (β₀)
# model.params["Property_Age"]           → β₁ (slope for age)
# model.params["Distance_to_Tech_Hub"]   → β₂ (slope for distance)
b0 = model.params["const"]
b1 = model.params["Property_Age"]
b2 = model.params["Distance_to_Tech_Hub"]

# ── 4. Build the regression surface over a grid ──────────────────────────────
# np.meshgrid creates a 2-D coordinate grid from two 1-D arrays.
# age_range and dist_range define the range of each axis.
# age_grid[i,j] and dist_grid[i,j] give the coordinates of every grid cell,
# letting us vectorise the prediction across the full surface in one step.
age_range  = np.linspace(df["Property_Age"].min(),         df["Property_Age"].max(),         40)
dist_range = np.linspace(df["Distance_to_Tech_Hub"].min(), df["Distance_to_Tech_Hub"].max(), 40)
age_grid, dist_grid = np.meshgrid(age_range, dist_range)

# Plug grid coordinates into the fitted equation: ŷ = β₀ + β₁·age + β₂·dist
price_surface = b0 + b1 * age_grid + b2 * dist_grid

# ── 5. Build the Plotly figure ───────────────────────────────────────────────
fig = go.Figure()

# Scatter: actual data points
fig.add_trace(go.Scatter3d(
    x=df["Property_Age"],
    y=df["Distance_to_Tech_Hub"],
    z=df["Sale_Price"],
    mode="markers",
    marker=dict(size=4, color=df["Sale_Price"],
                colorscale="Viridis", opacity=0.8),
    name="Observed"
))

# Surface: fitted hyperplane
fig.add_trace(go.Surface(
    x=age_grid,
    y=dist_grid,
    z=price_surface,
    colorscale="RdBu",
    opacity=0.6,
    name="Fitted plane"
))

fig.update_layout(
    title="OLS Hyperplane: Sale Price ~ Property Age + Distance to Tech Hub",
    scene=dict(
        xaxis_title="Property Age (yrs)",
        yaxis_title="Distance to Hub (km)",
        zaxis_title="Sale Price ($)",
    ),
    margin=dict(l=0, r=0, b=0, t=40),
)

fig.show()

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.926
Model:                            OLS   Adj. R-squared:                  0.925
Method:                 Least Squares   F-statistic:                     1235.
Date:                Wed, 18 Mar 2026   Prob (F-statistic):          3.50e-112
Time:                        20:09:08   Log-Likelihood:                -2305.9
No. Observations:                 200   AIC:                             4618.
Df Residuals:                     197   BIC:                             4628.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                 4.521e+05 